# F03: SQL Database OLTP Simulation

Generate financial domain data and simulate multi-day OLTP loads against a Fabric SQL Database using `FabricSqlDatabaseWriter`.

## What You'll Learn
- Generate financial domain data (accounts, transactions, ledger entries)
- Connect to a Fabric SQL Database with Azure CLI authentication
- Use `create_insert` mode for initial load (Day 1)
- Use `insert_only` and `append` modes to simulate daily incremental loads (Day 2+)
- Tune `batch_size` for optimal insert throughput

## Prerequisites
- Python 3.10+
- `sqllocks-spindle[fabric-sql]` installed
- ODBC Driver 18 for SQL Server
- A Fabric SQL Database endpoint (*.database.fabric.microsoft.com)
- `az login` completed

## Time Estimate
~15 minutes

In [1]:
# %pip install sqllocks-spindle[fabric-sql]

## Step 1: Generate Financial Domain Data (Day 1)

**What we're about to do:** Generate a financial dataset at small scale. The financial domain includes accounts, transactions, ledger entries, and GL accounts -- a realistic OLTP workload.

**Why this matters:** OLTP systems receive incremental writes daily. To test your pipeline, you need to simulate both the initial load and subsequent daily appends. Spindle lets you generate separate batches with different seeds to simulate this.

In [2]:
from sqllocks_spindle import Spindle, FinancialDomain

spindle = Spindle()

# Day 1: Initial full load
day1 = spindle.generate(domain=FinancialDomain(), scale="small", seed=42)
print("=== Day 1 (Initial Load) ===")
print(day1.summary())

# Inspect a sample table
print("\n--- Sample: first 5 rows of first table ---")
first_table = day1.table_names[0]
print(day1[first_table].head())

=== Day 1 (Initial Load) ===
Spindle Generation Result
Schema: financial_3nf
Domain: financial
Mode:   3nf
Seed:   42
Time:   0.3s

Table                             Rows  Columns
---------------------------------------------
branch                             200        9
customer                         1,000       10
account                          2,200        7
card                             1,760        8
loan                               400        9
loan_payment                     4,800        7
statement                       13,200        8
transaction_category                40        4
transaction                     10,000        9
fraud_flag                         200        6
---------------------------------------------
TOTAL                           33,800

--- Sample: first 5 rows of first table ---
   branch_id                branch_name             city  \
0          1       Market Street Branch           Casper   
1          2           Riverside Branch     

## Step 2: Configure the Writer for Fabric SQL Database

**What we're about to do:** Set up `FabricSqlDatabaseWriter` targeting a Fabric SQL Database. Note the server endpoint difference: SQL Database uses `*.database.fabric.microsoft.com` while Warehouse uses `*.datawarehouse.fabric.microsoft.com`.

**Key concept:** Fabric SQL Database supports full OLTP semantics (IDENTITY, PK constraints, indexes) unlike Fabric Warehouse. This makes it the right target for transactional simulation.

In [3]:
from sqllocks_spindle.fabric.sql_database_writer import FabricSqlDatabaseWriter

# Replace with your Fabric SQL Database endpoint
CONNECTION_STRING = (
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=YOUR_SERVER.database.fabric.microsoft.com;"
    "Database=YOUR_DATABASE;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
)

writer = FabricSqlDatabaseWriter(
    connection_string=CONNECTION_STRING,
    auth_method="cli",  # Uses az login token; switch to "msi" in Fabric Notebooks
)

print("Writer configured for Fabric SQL Database")
print(f"Auth method: cli (Azure CLI)")
print(f"Supported modes: create_insert, insert_only, truncate_insert, append")

Writer configured for Fabric SQL Database
Auth method: cli (Azure CLI)
Supported modes: create_insert, insert_only, truncate_insert, append


## Step 3: Simulate Day 1 Initial Load and Day 2 Incremental Append

**What we're about to do:** Demonstrate the two-phase loading pattern:
1. **Day 1** -- `create_insert` mode: drops tables, creates schema, inserts all rows
2. **Day 2** -- `append` mode: generates a new batch with a different seed and appends to existing tables

**Key concept:** Using different seeds produces different (but structurally identical) data, simulating new transactions arriving the next day. The `batch_size` parameter controls how many rows are sent per INSERT statement -- tune this for your network latency.

**Gotcha:** In `append` mode, Spindle does NOT check for duplicate PKs. Your schema should use UUIDs or sequences that won't collide across seeds.

In [4]:
# Day 2: Generate incremental data with a different seed
day2 = spindle.generate(domain=FinancialDomain(), scale="small", seed=99)
print("=== Day 2 (Incremental Load) ===")
print(day2.summary())

# Compare Day 1 vs Day 2 row counts
print("\n--- Day 1 vs Day 2 Comparison ---")
print(f"{'Table':<25} {'Day 1':>8} {'Day 2':>8}")
print("-" * 45)
for table_name in day1.table_names:
    d1_count = day1.row_counts.get(table_name, 0)
    d2_count = day2.row_counts.get(table_name, 0)
    print(f"{table_name:<25} {d1_count:>8,} {d2_count:>8,}")

=== Day 2 (Incremental Load) ===
Spindle Generation Result
Schema: financial_3nf
Domain: financial
Mode:   3nf
Seed:   99
Time:   0.1s

Table                             Rows  Columns
---------------------------------------------
branch                             200        9
customer                         1,000       10
account                          2,200        7
card                             1,760        8
loan                               400        9
loan_payment                     4,800        7
statement                       13,200        8
transaction_category                40        4
transaction                     10,000        9
fraud_flag                         200        6
---------------------------------------------
TOTAL                           33,800

--- Day 1 vs Day 2 Comparison ---
Table                        Day 1    Day 2
---------------------------------------------
branch                         200      200
customer                     1,000  

In [5]:
# Uncomment to execute against a live Fabric SQL Database:

# --- Day 1: Full reset ---
# result_day1 = writer.write(
#     day1,
#     schema_name="dbo",
#     mode="create_insert",
#     batch_size=1000,
# )
# print("Day 1 load:")
# print(result_day1.summary())

# --- Day 2: Append new data ---
# result_day2 = writer.write(
#     day2,
#     schema_name="dbo",
#     mode="append",         # INSERT without DROP/CREATE/TRUNCATE
#     batch_size=500,        # Smaller batches for lower latency
# )
# print("\nDay 2 append:")
# print(result_day2.summary())

# Batch size tuning guidance
print("Batch size tuning guide:")
print("  batch_size=100   - Low latency, more round trips (good for small tables)")
print("  batch_size=1000  - Balanced (default, recommended for most cases)")
print("  batch_size=5000  - High throughput, more memory (good for large fact tables)")

Batch size tuning guide:
  batch_size=100   - Low latency, more round trips (good for small tables)
  batch_size=1000  - Balanced (default, recommended for most cases)
  batch_size=5000  - High throughput, more memory (good for large fact tables)


## What's Next?

You've simulated a two-day OLTP loading pattern -- initial load plus incremental append -- against a Fabric SQL Database.

- **F05: Chaos Pipeline Testing** -- Inject data quality issues into your Day 2 data and test your pipeline's resilience
- **F02: Warehouse Dimensional Load** -- Load the same data as a star schema into a Fabric Warehouse for analytics
- **F01: Medallion Architecture** -- Build a full Bronze/Silver/Gold pipeline with validation gates